# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking
**Document:** *Dungeons & Dragons* 5e **System Reference Document 5.2.1** (`data/dnd_srd.pdf`, 364 pages).

Both pipelines run over the exact same chunks:
- **Pipeline 1 (baseline):** bi-encoder, cosine top-10, top-3 go to the LLM -> *Answer 1*
- **Pipeline 2 (re-rank):** the same top-10 are re-scored by a cross-encoder, top-3 go to the LLM (identical prompt) -> *Answer 2*

## 1. Setup

In [1]:
# pip install pypdf sentence-transformers requests pandas numpy
import os, re, time, getpass, statistics
import numpy as np, pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF_PATH   = "data/dnd_srd.pdf"
# Sections of the SRD that are indexed (PDF page numbers, inclusive)
SECTIONS   = [(5, 18, "Playing the Game"), (89, 103, "Equipment"), (176, 191, "Rules Glossary")]
CHUNK_SIZE = 200      # words per chunk
OVERLAP    = 40       # words shared between neighbouring chunks
BI_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
CE_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL  = "qwen/qwen3.8-27b"   # free on Groq (check console.groq.com for models your key can use)
TOP_K, TOP_N = 10, 3  # retrieve 10, give 3 to the LLM

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

## 2. Load, clean and chunk the PDF

In [2]:
def clean(text):
    text = re.sub(r"System Reference Document 5\.2\.1\s*\d*", " ", text)   # running page header
    text = re.sub(r"(\w)\s?-\n(\w)", r"\1\2", text)                       # re-join words broken across lines
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def build_chunks(path, sections):
    reader, chunks = PdfReader(path), []
    for first, last, name in sections:
        words = []                                       # (word, page)
        for page_no in range(first, last + 1):
            words += [(w, page_no) for w in clean(reader.pages[page_no - 1].extract_text() or "").split()]
        i = 0
        while i < len(words):
            seg = words[i:i + CHUNK_SIZE]
            chunks.append({"id": len(chunks), "section": name, "page": seg[0][1], "text": " ".join(w for w, _ in seg)})
            if i + CHUNK_SIZE >= len(words):
                break
            i += CHUNK_SIZE - OVERLAP
    return chunks

chunks = build_chunks(PDF_PATH, SECTIONS)
print(len(chunks), "chunks;", sum(len(c["text"].split()) for c in chunks), "words incl. overlap")
print(pd.Series([c["section"] for c in chunks]).value_counts().to_dict())
chunks[20]

188 chunks; 37384 words incl. overlap
{'Rules Glossary': 68, 'Playing the Game': 61, 'Equipment': 59}


{'id': 20,
 'section': 'Playing the Game',
 'page': 9,
 'text': 'mood and intentions. Intimidation Charisma Awe or threaten someone into doing what you want. Investigation Intelligence Find obscure information in books, or deduce how something works. Medicine Wisdom Diagnose an illness, or determine what killed the recently slain. Nature Intelligence Recall lore about terrain, plants, animals, and weather. Perception Wisdom Using a combination of senses, notice something that’s easy to miss. Performance Charisma Act, tell a story, perform music, or dance. Persuasion Charisma Honestly and graciously convince someone of something. Religion Intelligence Recall lore about gods, religious rituals, and holy symbols. Sleight of Hand Dexterity Pick a pocket, conceal a handheld object, or perform legerdemain. Stealth Dexterity Escape notice by moving quietly and hiding behind things. Survival Wisdom Follow tracks, forage, find a trail, or avoid natural hazards. Action Summary Disengage Your mov

## 3. Index with the bi-encoder

In [3]:
bi_encoder = SentenceTransformer(BI_MODEL)
cross_encoder = CrossEncoder(CE_MODEL)

# Normalised embeddings -> dot product == cosine similarity
chunk_emb = bi_encoder.encode([c["text"] for c in chunks], normalize_embeddings=True)
chunk_emb.shape

(188, 384)

## 4. The two retrieval pipelines

In [4]:
def baseline_retrieve(query, k=TOP_K):
    """Pipeline 1: bi-encoder cosine top-k. Returns chunk ids, best first."""
    q = bi_encoder.encode(query, normalize_embeddings=True)
    scores = chunk_emb @ q
    return [int(i) for i in np.argsort(-scores)[:k]]

def rerank(query, candidate_ids):
    """Pipeline 2: score each (query, chunk) pair jointly with the cross-encoder."""
    scores = cross_encoder.predict([(query, chunks[i]["text"]) for i in candidate_ids])
    order = np.argsort(-scores)
    return [candidate_ids[j] for j in order], [float(scores[j]) for j in order]

# warm-up so the first timed query does not include one-off model start-up cost
_ = rerank("warm up", baseline_retrieve("warm up"))

## 5. Answer generation (Groq)
Same prompt template for both pipelines; only the 3 context chunks differ. Temperature 0, so differences between Answer 1 and Answer 2 come from the context, not sampling.

In [5]:
import requests

PROMPT = """You are answering a question about the Dungeons & Dragons 5e rules using ONLY the rulebook excerpts below.
If the excerpts do not contain the answer, say so instead of guessing.

Rulebook excerpts:
{context}

Question: {question}
Answer concisely."""

# Load GROQ_API_KEY from a local .env file if present
if os.path.exists(".env"):
    for line in open(".env"):
        if "=" in line and not line.lstrip().startswith("#"):
            k, v = line.strip().split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

api_key = os.environ.get("GROQ_API_KEY")
if not api_key and not os.environ.get("SKIP_LLM"):
    api_key = getpass.getpass("GROQ_API_KEY (blank to skip generation): ")
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

def generate(question, chunk_ids, retries=5):
    if not api_key:
        return "(generation skipped: no API key)"
    context = "\n\n".join(f"[Excerpt {n+1}, page {chunks[i]['page']}] {chunks[i]['text']}"
                           for n, i in enumerate(chunk_ids))
    body = {"model": LLM_MODEL, "temperature": 0, "max_tokens": 400,
            "messages": [{"role": "user", "content": PROMPT.format(context=context, question=question)}]}
    for attempt in range(retries):
        try:
            r = requests.post(GROQ_URL, headers={"Authorization": f"Bearer {api_key}"}, json=body, timeout=90)
            if r.status_code == 429:                        # free-tier rate limit: honour Retry-After, then retry
                time.sleep(float(r.headers.get("retry-after", 20)))
                raise RuntimeError("rate limited")
            r.raise_for_status()
            time.sleep(10)                                  # ~1.2k tokens/call vs 8k tokens/min free limit -> stay under it
            text = r.json()["choices"][0]["message"]["content"] or ""
            return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
        except Exception as e:
            print(f"  LLM call failed ({type(e).__name__}: {str(e)[:120]}); retry {attempt+1}/{retries}")
            time.sleep(10 * (attempt + 1))
    return "(generation failed after retries)"

## 6. Experiment harness
Each test query carries a **gold phrase** (or a list of them when a rule is stated in more than one place): a short piece of rulebook text that must be in the chunk(s) that answer it. We look up which chunks contain it, then track their rank before and after re-ranking. A query counts as a *rescue* when the gold chunk is outside the baseline top-3 (rank 4-10) and inside the re-ranked top-3. A query with an empty gold phrase has no answer in the document at all.

In [6]:
def gold_ids(phrases):
    if not phrases:                      # out-of-scope question: the document has no answer chunk
        return []
    phrases = [phrases] if isinstance(phrases, str) else phrases
    return sorted({c["id"] for c in chunks for p in phrases if p.lower() in c["text"].lower()})

def best_rank(ranking, ids):
    ranks = [ranking.index(i) + 1 for i in ids if i in ranking]
    return min(ranks) if ranks else None

def run_query(question, gold_phrase, with_llm=True):
    ids = gold_ids(gold_phrase)

    t0 = time.perf_counter()
    base = baseline_retrieve(question)
    t_base = time.perf_counter() - t0

    t1 = time.perf_counter()
    reranked, ce_scores = rerank(question, base)
    t_ce = time.perf_counter() - t1

    out = {"question": question, "gold_chunks": ids,
           "base_rank": best_rank(base, ids), "rerank_rank": best_rank(reranked, ids),
           "base_top10": base, "rerank_top10": reranked,
           "t_baseline_ms": t_base * 1000, "t_cross_encoder_ms": t_ce * 1000,
           "t_rerank_total_ms": (t_base + t_ce) * 1000}
    if with_llm:
        out["answer1"] = generate(question, base[:TOP_N])
        out["answer2"] = generate(question, reranked[:TOP_N])
    return out

## 7. Test queries
Grouped by the failure mode each is meant to expose. Ranks are measured, not assumed; the table below shows what actually happened, including cases where re-ranking does not help or makes things worse.

In [7]:
QUERIES = [
    # (failure mode, question, gold phrase or list of phrases)
    ("Exception / corner case",  "Does casting a spell as a ritual use up a spell slot?", "doesn’t expend a spell slot"),
    ("Exception / corner case",  "Does moving via magical transportation trigger an opportunity attack?",
         ["don’t provoke an Opportunity Attack when you Teleport", "teleportation never provokes Opportunity Attacks"]),
    ("Exception / corner case",  "What happens if you have both advantage and disadvantage on the same roll?", "Advantage and Disadvantage on the same roll cancel each other"),
    ("Negation / constraint",    "Can healing give me more hit points than my maximum?", "Your Hit Points can’t exceed your Hit Point maximum"),
    ("Negation / constraint",    "Does a stable creature at 0 hit points still have to make death saving throws?", "A Stable creature doesn’t make Death Saving Throws"),
    ("Negation / constraint",    "What is the limitation on how often you can use the Cleave property in a single turn?", "You can make this extra attack only once per turn"),
    ("Keyword trap",             "How much does moving through rough ground increase the cost of each foot traveled?", "every foot of movement in that space costs 1 extra foot"),
    ("Buried chunk (other)",     "What specific action does the Game Master usually request when dice are involved in a social exchange?", "ask you to take the Influence action"),
    ("Buried chunk (other)",     "How much does half cover increase my armor class?", "+2 bonus to AC and Dexterity saving throws"),
    ("Re-ranking hurts",         "What penalty do you suffer on ability checks if you wear armor you are not proficient with?", "you have Disadvantage on any D20 Test"),
    ("Re-ranking hurts",         "How much damage does a makeshift object deal when it hits a target?", "the weapon deals 1d4 damage"),
    ("Control (easy)",           "What happens when I roll a natural 20 on a death saving throw?", "If you roll a 20 on the d20, you regain 1 Hit Point"),
    ("Control (easy)",           "Can I take another Reaction if I already used my Reaction this round?", "you can’t take another one until the start of your next turn"),
    # Not in the document at all: the right behaviour is to say so instead of guessing (no gold phrase)
    ("Not in document",          "How much does the D&D Player’s Handbook cost in dollars?", ""),
]

results = [dict(mode=m, **run_query(q, g)) for m, q, g in QUERIES]

summary = pd.DataFrame([{
    "failure mode": r["mode"], "question": r["question"][:70],
    "gold chunk(s)": r["gold_chunks"],
    "baseline rank": r["base_rank"], "re-ranked rank": r["rerank_rank"],
    "rescued (4-10 -> top3)": bool(r["base_rank"] and r["base_rank"] > TOP_N and r["rerank_rank"] and r["rerank_rank"] <= TOP_N),
} for r in results])
summary

,failure mode,question,gold chunk(s),baseline rank,re-ranked rank,rescued (4-10 -> top3)
0,Exception / corner case,Does casting a spell as a ritual use up a spell slot?,"[169, 170]",1.0,1.0,False
1,Exception / corner case,Does moving via magical transportation trigger an opportunity attack?,"[47, 183]",1.0,1.0,False
2,Exception / corner case,What happens if you have both advantage and disadvantage on the same r,"[122, 143]",4.0,2.0,True
3,Negation / constraint,Can healing give me more hit points than my maximum?,[55],1.0,3.0,False
4,Negation / constraint,Does a stable creature at 0 hit points still have to make death saving,"[58, 59]",1.0,1.0,False
5,Negation / constraint,What is the limitation on how often you can use the Cleave property in,[67],6.0,1.0,True
6,Keyword trap,How much does moving through rough ground increase the cost of each fo,[142],4.0,1.0,True
7,Buried chunk (other),What specific action does the Game Master usually request when dice ar,[26],8.0,3.0,True
8,Buried chunk (other),How much does half cover increase my armor class?,"[45, 134]",4.0,2.0,True
9,Re-ranking hurts,What penalty do you suffer on ability checks if you wear armor you are,"[73, 124, 125]",4.0,9.0,False


## 8. Answer 1 (baseline) vs. Answer 2 (re-ranked)

In [8]:
for r in results:
    print("=" * 100)
    print(f"[{r['mode']}] {r['question']}")
    print(f"gold chunk(s) {r['gold_chunks']} | baseline rank {r['base_rank']} -> re-ranked rank {r['rerank_rank']}")
    print("baseline top-3 :", r["base_top10"][:TOP_N], "| re-ranked top-3:", r["rerank_top10"][:TOP_N])
    print("\nAnswer 1 (baseline):\n", r["answer1"])
    print("\nAnswer 2 (re-ranked):\n", r["answer2"])

[Exception / corner case] Does casting a spell as a ritual use up a spell slot?
gold chunk(s) [169, 170] | baseline rank 1 -> re-ranked rank 1
baseline top-3 : [170, 169, 118] | re-ranked top-3: [170, 169, 175]

Answer 1 (baseline):
 No, the ritual version of a spell does not expend a spell slot.

Answer 2 (re-ranked):
 No, the ritual version of a spell does not expend a spell slot.
[Exception / corner case] Does moving via magical transportation trigger an opportunity attack?
gold chunk(s) [47, 183] | baseline rank 1 -> re-ranked rank 1
baseline top-3 : [47, 167, 143] | re-ranked top-3: [183, 47, 168]

Answer 1 (baseline):
 No. You do not provoke an Opportunity Attack when you Teleport.

Answer 2 (re-ranked):
 No. Teleportation never provokes Opportunity Attacks, and you do not provoke an Opportunity Attack when you are moved without using your movement, action, Bonus Action, or Reaction.
[Exception / corner case] What happens if you have both advantage and disadvantage on the same ro

## 8.1 Gold rules, answers and verdicts

For every test question: the rulebook rule that answers it (the *gold rule*, tidied from the PDF text), the exact answer each pipeline gave in section 8, and my own verdict after comparing the answer with the rulebook.

### Q1. Does casting a spell as a ritual use up a spell slot?
- **Gold rule** (chunks 169-170, p.187): "If you have a spell prepared that has the Ritual tag, you can cast that spell as a Ritual. The Ritual version of a spell takes 10 minutes longer to cast than normal. It also doesn’t expend a spell slot."
- **Answer 1 (baseline)** - *Correct*: No, the ritual version of a spell does not expend a spell slot.
- **Answer 2 (re-ranked)** - *Correct*: No, the ritual version of a spell does not expend a spell slot.

### Q2. Does moving via magical transportation trigger an opportunity attack?
- **Gold rule** (chunks 47 and 183, pp.15 and 190): "You also don’t provoke an Opportunity Attack when you Teleport or when you are moved without using your movement, action, Bonus Action, or Reaction. (Also: teleportation never provokes Opportunity Attacks.)"
- **Answer 1 (baseline)** - *Correct*: No. You do not provoke an Opportunity Attack when you Teleport.
- **Answer 2 (re-ranked)** - *Correct*: No. Teleportation never provokes Opportunity Attacks, and you do not provoke an Opportunity Attack when you are moved without using your movement, action, Bonus Action, or Reaction.

### Q3. What happens if you have both advantage and disadvantage on the same roll?
- **Gold rule** (chunks 122 and 143, pp.176-177): "A roll can’t be affected by more than one Advantage, and Advantage and Disadvantage on the same roll cancel each other."
- **Answer 1 (baseline)** - *Correct*: The roll has neither Advantage nor Disadvantage, and you roll one d20.
- **Answer 2 (re-ranked)** - *Correct*: They cancel each other out, and you roll one d20.

### Q4. Can healing give me more hit points than my maximum?
- **Gold rule** (chunk 55, p.17): "Your Hit Points can’t exceed your Hit Point maximum, so any Hit Points regained in excess of the maximum are lost."
- **Answer 1 (baseline)** - *Correct*: No. Your Hit Points can’t exceed your Hit Point maximum, so any Hit Points regained in excess of the maximum are lost.
- **Answer 2 (re-ranked)** - *Correct*: No. Your Hit Points can’t exceed your Hit Point maximum, so any Hit Points regained in excess of the maximum are lost.

### Q5. Does a stable creature at 0 hit points still have to make death saving throws?
- **Gold rule** (chunk 58, p.17): "A Stable creature doesn’t make Death Saving Throws even though it has 0 Hit Points, but it still has the Unconscious condition."
- **Answer 1 (baseline)** - *Correct*: No, a Stable creature does not make Death Saving Throws.
- **Answer 2 (re-ranked)** - *Correct*: No, a Stable creature does not make Death Saving Throws.

### Q6. What is the limitation on how often you can use the Cleave property in a single turn?
- **Gold rule** (chunk 67, p.90): "Cleave: ... you can make a melee attack roll with the weapon against a second creature within 5 feet of the first ... You can make this extra attack only once per turn."
- **Answer 1 (baseline)** - *No answer (said the excerpts do not contain it, but they do)*: The provided rulebook excerpts do not contain information about the limitation on how often the Cleave property can be used in a single turn.
- **Answer 2 (re-ranked)** - *Correct*: You can make this extra attack only once per turn.

### Q7. How much does moving through rough ground increase the cost of each foot traveled?
- **Gold rule** (chunk 142, p.181): "If a space is Difficult Terrain, every foot of movement in that space costs 1 extra foot."
- **Answer 1 (baseline)** - *Correct*: 1 extra foot
- **Answer 2 (re-ranked)** - *Correct*: 1 extra foot

### Q8. What specific action does the Game Master usually request when dice are involved in a social exchange?
- **Gold rule** (chunk 26, p.11): "In such situations, the GM will typically ask you to take the Influence action."
- **Answer 1 (baseline)** - *No answer (said the excerpts do not contain it, but they do)*: The provided rulebook excerpts do not contain the answer to this question.
- **Answer 2 (re-ranked)** - *Correct*: The GM will typically ask you to take the **Influence action**.

### Q9. How much does half cover increase my armor class?
- **Gold rule** (chunk 45, p.15): "Cover table, Half: +2 bonus to AC and Dexterity saving throws (offered by another creature or an object that covers at least half of the target)."
- **Answer 1 (baseline)** - *No answer (said the excerpts do not contain it, but they do)*: The provided rulebook excerpts do not contain information about how much half cover increases Armor Class.
- **Answer 2 (re-ranked)** - *Correct*: +2 bonus to AC

### Q10. What penalty do you suffer on ability checks if you wear armor you are not proficient with?
- **Gold rule** (chunk 73, p.92): "If you wear Light, Medium, or Heavy armor and lack training with it, you have Disadvantage on any D20 Test that involves Strength or Dexterity, and you can’t cast spells."
- **Answer 1 (baseline)** - *No answer (said the excerpts do not contain it, but they do)*: Based on the provided excerpts, the text does not contain the answer. Excerpt 3 states that "only those with training can use them effectively" and begins a section titled "Light, Medium, or Heavy Armor If you wear Light, Medium, or Heavy armor and lack training with it," but the excerpt cuts off before specifying the actual penalty or disadvantage applied to ability checks.
- **Answer 2 (re-ranked)** - *No answer (said the excerpts do not contain it, but they do)*: The provided rulebook excerpts do not contain the answer to this question.

### Q11. How much damage does a makeshift object deal when it hits a target?
- **Gold rule** (chunk 154, p.183): "On a hit, the weapon deals 1d4 damage of a type the GM thinks is appropriate for the object."
- **Answer 1 (baseline)** - *No answer (said the excerpts do not contain it, but they do)*: The provided rulebook excerpts do not contain information about the damage dealt by makeshift objects.
- **Answer 2 (re-ranked)** - *No answer (said the excerpts do not contain it, but they do)*: The provided rulebook excerpts do not contain information about the damage dealt by a makeshift object.

### Q12. What happens when I roll a natural 20 on a death saving throw?
- **Gold rule** (chunk 58, p.17): "If you roll a 20 on the d20, you regain 1 Hit Point."
- **Answer 1 (baseline)** - *Correct*: You regain 1 Hit Point.
- **Answer 2 (re-ranked)** - *Correct*: You regain 1 Hit Point.

### Q13. Can I take another Reaction if I already used my Reaction this round?
- **Gold rule** (chunk 23, p.10): "When you take a Reaction, you can’t take another one until the start of your next turn."
- **Answer 1 (baseline)** - *Correct*: No. According to the rules, when you take a Reaction, you can’t take another one until the start of your next turn.
- **Answer 2 (re-ranked)** - *Correct*: No. Once you take a Reaction, you can’t take another one until the start of your next turn.

### Q14. How much does the D&D Player’s Handbook cost in dollars?
- **Gold rule:** none. The document never mentions the price, so the correct behaviour is to say the information is missing.
- **Answer 1 (baseline)** - *Correct (correctly said the information is not in the document)*: The provided rulebook excerpts do not contain information about the retail price of the D&D Player’s Handbook in dollars.
- **Answer 2 (re-ranked)** - *Correct (correctly said the information is not in the document)*: The provided rulebook excerpts do not contain information about the retail price of the D&D Player’s Handbook in dollars.

**Totals:** Answer 1 = 9 correct, 5 no answer, 0 wrong. Answer 2 = 12 correct, 2 no answer, 0 wrong. Neither pipeline gave a confidently wrong answer; the failures are all cases where the model said the answer was missing because the key chunk was not in its top 3. Q10 and Q11 fail in both pipelines.

## 9. Latency profiling
Retrieval-only timings, averaged over repeated runs of every test query (LLM time is excluded because it is identical in both pipelines and dominated by the network). *Re-rank total* = baseline retrieval + cross-encoder scoring of 10 pairs.

In [9]:
REPEATS = 5
rows = []
for _, q, g in QUERIES:
    runs = [run_query(q, g, with_llm=False) for _ in range(REPEATS)]
    rows.append({"baseline_ms": statistics.mean(r["t_baseline_ms"] for r in runs),
                 "cross_encoder_ms": statistics.mean(r["t_cross_encoder_ms"] for r in runs),
                 "rerank_total_ms": statistics.mean(r["t_rerank_total_ms"] for r in runs)})
lat = pd.DataFrame(rows, index=[q[:45] for _, q, _ in QUERIES]).round(1)
lat.loc["MEAN"] = lat.mean().round(1)
lat["overhead_x"] = (lat["rerank_total_ms"] / lat["baseline_ms"]).round(1)
lat

,baseline_ms,cross_encoder_ms,rerank_total_ms,overhead_x
Does casting a spell as a ritual use up a spe,11.3,74.5,85.8,7.6
Does moving via magical transportation trigge,8.1,71.9,80.0,9.9
What happens if you have both advantage and d,7.8,70.9,78.7,10.1
Can healing give me more hit points than my m,8.3,67.1,75.4,9.1
Does a stable creature at 0 hit points still,8.6,64.1,72.8,8.5
What is the limitation on how often you can u,9.6,97.0,106.5,11.1
How much does moving through rough ground inc,10.7,70.2,80.9,7.6
What specific action does the Game Master usu,9.2,73.5,82.7,9.0
How much does half cover increase my armor cl,7.8,65.4,73.2,9.4
What penalty do you suffer on ability checks,8.9,60.9,69.8,7.8
